# Store Sales — prévision des ventes

**Projet Kaggle · Store Sales - Time Series Forecasting**

Ce notebook présente le pipeline retenu pour prévoir les ventes quotidiennes de magasins et familles de produits en Équateur. Le meilleur score public rapporté pour le mélange présenté ici est **0,40665 RMSLE**, contre **0,41191** pour la soumission précédente. Cette soumission a atteint le **104e rang sur 1 520 participants** (environ le top **6,8 %**). Ces résultats proviennent du leaderboard Kaggle et ne sont pas reproduits par une exécution locale de ce notebook.

## Résumé du projet

- **Cible :** ventes quotidiennes (`sales`), avec apprentissage en `log1p(sales)`.
- **Validation :** fenêtres temporelles historiques, sans mélange aléatoire des dates.
- **Modèles :** LightGBM direct sur CPU et XGBoost à deux étapes sur GPU.
- **Mélange soumis :** 25 % LightGBM direct + 75 % XGBoost à deux étapes, combinés dans l'espace logarithmique.
- **Sortie :** fichier CSV de soumission distinct, avec contrôle des identifiants et des prédictions.

## Parcours de lecture

1. Charger et fusionner les fichiers de la compétition.
2. Construire les variables de calendrier, transactions, promotions et historique des ventes.
3. Entraîner les modèles et les comparer sur des fenêtres temporelles.
4. Réentraîner sur toutes les données étiquetées et générer le fichier de soumission.

> **Reproductibilité :** exécuter les cellules dans l'ordre. Dans Kaggle, activer l'accélérateur GPU avant l'exécution. Les résultats de validation ci-dessous sont des résultats observés sur Kaggle ; le score public dépend du jeu de test et peut varier.

## Résultats et sélection du modèle

| Variante | RMSLE OOF (7 fenêtres) | Score public Kaggle |
|---|---:|---:|
| LightGBM direct (CPU) | 0,412539 | — |
| XGBoost à deux étapes (GPU) | 0,409131 | — |
| Mélange 25 % LightGBM / 75 % XGBoost | 0,407937 | **0,40665** |
| Mélange 35 % LightGBM / 65 % XGBoost | **0,407842** | non soumis |

Le poids à 65 % XGBoost obtient le plus faible RMSLE OOF parmi les poids testés, mais ne gagne que 2 fenêtres sur 7 face au poids à 75 %. Le mélange à 75 % reste donc la version retenue ici : c'est celui qui a produit le meilleur score public rapporté et le classement 104/1 520. Les périodes de validation ont aussi servi à comparer les poids ; leur optimum n'est pas un estimateur indépendant de la performance future.

## 1. Chargement et assemblage des données

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Repère les fichiers de la compétition dans Kaggle ou dans le dossier du projet local.
required_data_files = [
    'train.csv', 'test.csv', 'oil.csv', 'stores.csv',
    'holidays_events.csv', 'transactions.csv'
]
candidate_data_dirs = [
    os.getcwd(),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), 'store_sales_project'),
    '/kaggle/input/store-sales-time-series-forecasting'
]
candidate_data_dirs.extend(
    os.path.dirname(path)
    for path in glob.glob('/kaggle/input/**/train.csv', recursive=True)
)
DATA_PATH = next(
    (
        directory for directory in dict.fromkeys(candidate_data_dirs)
        if all(os.path.isfile(os.path.join(directory, name)) for name in required_data_files)
    ),
    None
)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Fichiers de la compétition introuvables. Ajoute le dataset Kaggle en entrée '
        'ou place les six CSV requis dans le dossier de travail.'
    )
print(f'📁 Dossier des données : {DATA_PATH}')

print("📂 Chargement des données...")
train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'), parse_dates=['date'])
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'), parse_dates=['date'])
oil = pd.read_csv(os.path.join(DATA_PATH, 'oil.csv'), parse_dates=['date'])
stores = pd.read_csv(os.path.join(DATA_PATH, 'stores.csv'))
holidays = pd.read_csv(os.path.join(DATA_PATH, 'holidays_events.csv'), parse_dates=['date'])
transactions = pd.read_csv(os.path.join(DATA_PATH, 'transactions.csv'), parse_dates=['date'])    

# 1. Traitement propre du pétrole (interpolation linéaire + remplissage des bords)
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear').bfill().ffill()

# 2. Traitement propre des jours fériés (on ignore les jours transférés pour éviter les faux positifs)
holidays_clean = holidays[~(holidays['transferred'] == True)].copy()
holidays_clean['is_holiday'] = 1
# On garde une seule ligne par date pour éviter de dupliquer les lignes du train/test lors du merge
holidays_agg = holidays_clean[['date', 'is_holiday']].drop_duplicates(subset=['date'])

print("🔗 Fusion des tables...")
test['sales'] = np.nan
df = pd.concat([train, test], ignore_index=True)

# Fusions successives
df = df.merge(stores, on='store_nbr', how='left')
df = df.merge(oil, on='date', how='left')
df = df.merge(holidays_agg, on='date', how='left')

# Remplissage des NaN pour les jours non fériés
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)

# 3. TRI INDISPENSABLE avant de calculer les lags et rollings
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f"✅ Dimensions du DataFrame combiné : {df.shape}")
print("Aperçu des premières lignes :")
print(df.head(3))

## 2. Calendrier : jours fériés et séisme

In [ ]:
print("🎉 Intégration des jours fériés nationaux et du choc séisme 2016...")

# Conversion des colonnes date au bon format
df['date'] = pd.to_datetime(df['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

# 1. Gestion propre des jours fériés nationaux
holidays_clean = holidays[~(holidays['transferred'] == True)].copy()
national_holidays = holidays_clean[holidays_clean['locale'] == 'National'].drop_duplicates(subset=['date']).copy()
national_holidays['is_national_holiday'] = 1

df = df.drop(columns=['is_national_holiday'], errors='ignore')
df = df.merge(national_holidays[['date', 'is_national_holiday']], on='date', how='left')
df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)

# 2. Variable spécifique pour le séisme d'avril 2016 (Équateur)
df['earthquake_impact'] = 0
earthquake_start = pd.to_datetime('2016-04-16')
earthquake_end = pd.to_datetime('2016-06-30')
df.loc[(df['date'] >= earthquake_start) & (df['date'] <= earthquake_end), 'earthquake_impact'] = 1

print("✅ Choc séisme et jours fériés nationaux intégrés avec succès !")

## 3. Features de transactions

In [ ]:
print("📊 Traitement rigoureux des transactions (Configuration corrigée et sécurisée)...")

# 0. Sécurité : s'assurer que toutes les dates sont au bon format
df['date'] = pd.to_datetime(df['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

# 1. Grille continue par magasin
unique_dates = df[['date']].drop_duplicates()
unique_stores = stores[['store_nbr']]
full_store_grid = pd.MultiIndex.from_product(
    [unique_dates['date'], unique_stores['store_nbr']],
    names=['date', 'store_nbr']
).to_frame(index=False)

trans_clean = full_store_grid.merge(transactions, on=['date', 'store_nbr'], how='left')
trans_clean = trans_clean.sort_values(['store_nbr', 'date']).reset_index(drop=True)

# Interpolation douce, magasin par magasin
trans_clean['transactions'] = trans_clean.groupby('store_nbr')['transactions'].transform(
    lambda x: x.interpolate(method='linear').bfill().ffill()
)

# 2. Lags et rollings stricts (>= 16 jours, isolés par magasin)
trans_lags = [16, 17, 30, 60, 365]
for lag in trans_lags:
    trans_clean[f'trans_lag_{lag}'] = trans_clean.groupby('store_nbr')['transactions'].shift(lag)

trans_rolling_windows = [7, 14, 30]
for w in trans_rolling_windows:
    trans_clean[f'trans_rolling_{w}'] = trans_clean.groupby('store_nbr')['transactions'].transform(
        lambda x: x.shift(16).rolling(w).mean()
    )

# 3. Volatilité et momentum
for w in [7, 14]:
    trans_clean[f'trans_rolling_std_{w}'] = trans_clean.groupby('store_nbr')['transactions'].transform(
        lambda x: x.shift(16).rolling(w).std()
    )

trans_clean['trans_trend_ratio'] = trans_clean['trans_lag_16'] / (trans_clean['trans_lag_30'] + 1e-5)

# 4. Fusion propre
extra_trans_cols = [f'trans_rolling_std_{w}' for w in [7, 14]] + ['trans_trend_ratio']
cols_to_merge = ['date', 'store_nbr', 'transactions'] + \
                [f'trans_lag_{l}' for l in trans_lags] + \
                [f'trans_rolling_{w}' for w in trans_rolling_windows] + \
                extra_trans_cols

# Anti-collision
cols_to_drop = [c for c in cols_to_merge if c in df.columns and c not in ['date', 'store_nbr']]
df = df.drop(columns=cols_to_drop, errors='ignore')

n_before = len(df)
df = df.merge(trans_clean[cols_to_merge], on=['date', 'store_nbr'], how='left')
assert len(df) == n_before, "Le merge a dupliqué des lignes !"

print("✅ Grille de transactions corrigée et fusionnée avec succès !")

In [ ]:
print("📊 Axe 2 : Lags, Rollings et Totaux...")

# 0. Nettoyage anti-collision
cols_to_drop_promo = [c for c in df.columns if c.startswith('promo_') or c in ['store_total_promos', 'family_total_promos']]
df = df.drop(columns=cols_to_drop_promo, errors='ignore')

# 1. Lags sur onpromotion
promo_lags = [1, 7, 14, 16]
for lag in promo_lags:
    df[f'promo_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].shift(lag)

# 2. Rolling sur onpromotion (avec transform pour garantir l'intégrité des groupes)
promo_windows = [7, 14]
for w in promo_windows:
    df[f'promo_rolling_{w}'] = (
        df.groupby(['store_nbr', 'family'])['onpromotion']
        .transform(lambda x: x.shift(1).rolling(w).mean())
    )

# 3. Intensité magasin : Somme totale des produits en promo dans ce magasin à cette date
store_promo_agg = df.groupby(['date', 'store_nbr'])['onpromotion'].sum().reset_index()
store_promo_agg = store_promo_agg.rename(columns={'onpromotion': 'store_total_promos'})
df = df.merge(store_promo_agg, on=['date', 'store_nbr'], how='left')

# 4. Intensité famille : Somme de cette famille en promo tous magasins confondus
family_promo_agg = df.groupby(['date', 'family'])['onpromotion'].sum().reset_index()
family_promo_agg = family_promo_agg.rename(columns={'onpromotion': 'family_total_promos'})
df = df.merge(family_promo_agg, on=['date', 'family'], how='left')

print("✅ Features de promotion intégrées avec succès !")

## 4. Features de ventes et variables temporelles

In [ ]:
print("⚙️ Feature Engineering & Assemblage des transformations de base...")

# 0. SÉCURITÉ ABSOLUE : Tri chronologique strict pour garantir l'intégrité de tous les shifts et rollings
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
df['date'] = pd.to_datetime(df['date'])

# 1. Variables temporelles de base
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
df['is_payday'] = ((df['day'] == 15) | (df['date'].dt.is_month_end)).astype(int)

# 2. Cible log
df['sales_log'] = np.log1p(df['sales'])

# 3. Lags de ventes sécurisés (>= 16 jours)
lags = [16, 21, 28, 30, 60, 90, 365]
for lag in lags:
    df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales_log'].shift(lag)

# 4. Rolling ventes
rolling_windows = [14, 30]
for w in rolling_windows:
    df[f'rolling_mean_{w}'] = df.groupby(['store_nbr', 'family'])['sales_log'].transform(
        lambda x: x.shift(16).rolling(w).mean()
    )

# 5. Promos passées et rollings de promo propres
promo_lags = [1, 7, 14, 16]
for l in promo_lags:
    df[f'promo_lag_{l}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].shift(l)

for w in [7, 14]:
    df[f'promo_rolling_{w}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].transform(
        lambda x: x.shift(1).rolling(w).mean()
    )

# 6. Encodage des catégories pour LightGBM
cat_cols = ['store_nbr', 'family', 'city', 'state', 'type']
for col in cat_cols:
    df[col] = df[col].astype('category')

print("✅ Transformations de base et features de ventes appliquées avec succès !")

## 5. Assemblage des variables explicatives

In [ ]:
print("📦 Réassemblage de la liste finale des features (sans encodage cible global)...")

features = cat_cols + [
    'onpromotion', 'dcoilwtico', 'cluster', 'is_holiday', 'is_payday',
    'is_national_holiday', 'earthquake_impact',
    'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'store_total_promos', 'family_total_promos'
] + \
[f'lag_{l}' for l in lags] + \
[f'rolling_mean_{w}' for w in rolling_windows] + \
[f'promo_lag_{l}' for l in promo_lags] + \
['promo_rolling_7', 'promo_rolling_14'] + \
[f'trans_lag_{l}' for l in [16, 17, 30, 60, 365]] + \
[f'trans_rolling_{w}' for w in [7, 14, 30]] + \
[f'trans_rolling_std_{w}' for w in [7, 14]] + \
['trans_trend_ratio']

print(f"🎯 Total de features prêtes : {len(features)}")

In [ ]:
print("ℹ️ L'encodage cible DOW global est désactivé pour éviter les fuites temporelles.")

## 6. Configuration GPU et validation étendue

Avant l'exécution, active **Settings → Accelerator → GPU** dans Kaggle. XGBoost utilise `tree_method='hist'` et `device='cuda'` ; LightGBM reste la référence CPU. Le modèle occupe un GPU à la fois pour limiter la mémoire.

In [ ]:
import lightgbm as lgb
import xgboost as xgb

XGB_DEVICE = 'cuda'
XGB_COMMON_PARAMS = {
    'tree_method': 'hist',
    'device': XGB_DEVICE,
    'enable_categorical': True,
    'max_depth': 8,
    'min_child_weight': 150,
    'learning_rate': 0.025,
    'subsample': 0.75,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'max_bin': 256,
    'n_jobs': 4,
    'verbosity': 0
}

xgboost_build = xgb.build_info()
if not xgboost_build.get('USE_CUDA', False):
    raise RuntimeError(
        'Cette cellule requiert une version XGBoost compilée avec CUDA. '
        'Active un accélérateur GPU Kaggle puis vérifie la version XGBoost.'
    )

print(f"✅ XGBoost {xgb.__version__} prêt pour l'entraînement GPU ({XGB_DEVICE}).")
print("Note : min_child_weight=150 adapte approximativement min_child_samples=150 de LightGBM ; les deux paramètres ne sont pas équivalents.")

## 7. Validation temporelle et comparaison des mélanges

La validation compare LightGBM CPU à XGBoost GPU sur sept fenêtres historiques. Les itérations sont fixées à partir des essais antérieurs. Un balayage des poids examine les compromis entre le RMSLE global et la stabilité par fenêtre ; le modèle final reste fixé au mélange 25/75 qui a obtenu 0,40665 sur Kaggle.

In [ ]:
print("🧭 Validation temporelle élargie : régression directe vs modèle à deux étapes...")

# Fenêtres de 16 jours non chevauchantes, réparties sur plusieurs saisons.
extended_val_cutoffs = pd.to_datetime([
    '2016-10-01', '2016-12-01', '2017-02-01', '2017-04-01',
    '2017-06-01', '2017-07-01', '2017-08-01'
])

# Itérations fixées à partir des médianes de la CV précédente ; aucune
# early stopping n'est faite sur les fenêtres évaluées ici.
extended_baseline_iterations = 2918
extended_classifier_iterations = 1075
extended_regressor_iterations = 3707
extended_fold_results = []
extended_oof_parts = []

for fold_index, cutoff in enumerate(extended_val_cutoffs):
    validation_end = cutoff + pd.Timedelta(days=15)
    train_data = df.loc[
        (df['date'] < cutoff) & df['sales'].notna()
    ].copy()
    validation_data = df.loc[
        (df['date'] >= cutoff)
        & (df['date'] <= validation_end)
        & df['sales'].notna()
    ].copy()
    if train_data.empty or validation_data.empty:
        raise ValueError(f"Données d'entraînement ou de validation absentes pour {cutoff:%Y-%m-%d}.")

    print(
        f"\nFold {fold_index + 1}/{len(extended_val_cutoffs)} | "
        f"{cutoff:%Y-%m-%d} au {validation_end:%Y-%m-%d} | "
        f"{validation_data['date'].nunique()} jours étiquetés, "
        f"{len(validation_data):,} lignes"
    )
    X_train = train_data[features]
    X_validation = validation_data[features]
    y_train_nonzero = train_data['sales'].gt(0).astype('int8')
    positive_train = train_data['sales'].gt(0)
    if y_train_nonzero.nunique() != 2 or not positive_train.any():
        raise ValueError(f"Classes insuffisantes pour entraîner le fold {fold_index + 1}.")

    baseline_model = lgb.LGBMRegressor(
        objective='regression_l2',
        metric='rmse',
        boosting_type='gbdt',
        n_estimators=extended_baseline_iterations,
        learning_rate=0.025,
        num_leaves=63,
        max_depth=8,
        min_child_samples=150,
        subsample=0.75,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42 + fold_index,
        n_jobs=-1,
        verbose=-1
    )
    baseline_model.fit(X_train, train_data['sales_log'])
    baseline_log_prediction = np.maximum(
        baseline_model.predict(X_validation), 0
    )

    classifier = xgb.XGBClassifier(
        **XGB_COMMON_PARAMS,
        n_estimators=extended_classifier_iterations,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42 + fold_index
    )
    classifier.fit(X_train, y_train_nonzero)

    regressor = xgb.XGBRegressor(
        **XGB_COMMON_PARAMS,
        n_estimators=extended_regressor_iterations,
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42 + fold_index
    )
    regressor.fit(
        X_train.loc[positive_train],
        train_data.loc[positive_train, 'sales_log']
    )
    classifier.set_params(device='cpu')
    regressor.set_params(device='cpu')
    probability_nonzero = classifier.predict_proba(X_validation)[:, 1]
    conditional_log_sales = np.maximum(regressor.predict(X_validation), 0)
    two_stage_log_prediction = probability_nonzero * conditional_log_sales
    actual_log_sales = validation_data['sales_log'].to_numpy()

    fold_result = pd.DataFrame({
        'id': validation_data['id'].to_numpy(),
        'date': validation_data['date'].to_numpy(),
        'fold': fold_index + 1,
        'baseline_squared_log_error': (
            baseline_log_prediction - actual_log_sales
        ) ** 2,
        'two_stage_squared_log_error': (
            two_stage_log_prediction - actual_log_sales
        ) ** 2,
        'actual_log_sales': actual_log_sales,
        'baseline_log_prediction': baseline_log_prediction,
        'two_stage_log_prediction': two_stage_log_prediction,
        'probability_nonzero': probability_nonzero,
        'actual_nonzero': validation_data['sales'].gt(0).astype('int8').to_numpy()
    })
    extended_oof_parts.append(fold_result)
    extended_fold_results.append({
        'fold': fold_index + 1,
        'cutoff': cutoff,
        'n_days': fold_result['date'].nunique(),
        'n': len(fold_result),
        'baseline_mse': fold_result['baseline_squared_log_error'].mean(),
        'two_stage_mse': fold_result['two_stage_squared_log_error'].mean()
    })

extended_oof = pd.concat(extended_oof_parts, ignore_index=True)
if not extended_oof['id'].is_unique:
    raise ValueError("Les fenêtres temporelles élargies se chevauchent sur les identifiants.")

extended_fold_summary = pd.DataFrame(extended_fold_results)
extended_fold_summary['baseline_rmsle'] = np.sqrt(extended_fold_summary.pop('baseline_mse'))
extended_fold_summary['two_stage_rmsle'] = np.sqrt(extended_fold_summary.pop('two_stage_mse'))
extended_fold_summary['delta_rmsle'] = (
    extended_fold_summary['two_stage_rmsle']
    - extended_fold_summary['baseline_rmsle']
)
print("\n📊 RMSLE par fenêtre (delta négatif = avantage du modèle deux étapes) :")
display(extended_fold_summary)

baseline_global_rmsle = np.sqrt(extended_oof['baseline_squared_log_error'].mean())
two_stage_global_rmsle = np.sqrt(extended_oof['two_stage_squared_log_error'].mean())
print("\n🏁 Résumé OOF élargi, pondéré par le nombre de lignes :")
display(pd.DataFrame([{
    'n_fenetres': len(extended_fold_summary),
    'n_predictions': len(extended_oof),
    'baseline_rmsle': baseline_global_rmsle,
    'two_stage_rmsle': two_stage_global_rmsle,
    'delta_rmsle': two_stage_global_rmsle - baseline_global_rmsle,
    'fenetres_gagnees_par_deux_etapes': int(
        extended_fold_summary['delta_rmsle'].lt(0).sum()
    )
}]))

blend_rows = []
blend_two_stage_weights = [0.0, 0.25, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
reference_blend_two_stage_weight = 0.75
reference_blend_log_prediction = (
    (1 - reference_blend_two_stage_weight) * extended_oof['baseline_log_prediction']
    + reference_blend_two_stage_weight * extended_oof['two_stage_log_prediction']
)
reference_blend_fold_mse = (
    (reference_blend_log_prediction - extended_oof['actual_log_sales']) ** 2
).groupby(extended_oof['fold']).mean()
for two_stage_weight in blend_two_stage_weights:
    blend_log_prediction = (
        (1 - two_stage_weight) * extended_oof['baseline_log_prediction']
        + two_stage_weight * extended_oof['two_stage_log_prediction']
    )
    blend_squared_error = (
        blend_log_prediction - extended_oof['actual_log_sales']
    ) ** 2
    blend_oof = extended_oof[['fold']].copy()
    blend_oof['squared_log_error'] = blend_squared_error
    fold_blend = blend_oof.groupby('fold')['squared_log_error'].mean()
    blend_rows.append({
        'poids_deux_etapes': two_stage_weight,
        'poids_direct': 1 - two_stage_weight,
        'rmsle_global': np.sqrt(blend_squared_error.mean()),
        'fenetres_gagnees_vs_deux_etapes': int(
            (fold_blend < extended_oof.groupby('fold')['two_stage_squared_log_error'].mean()).sum()
        ),
        'fenetres_gagnees_vs_75pct': int((fold_blend < reference_blend_fold_mse).sum()),
        **{
            f'rmsle_fold_{fold_number}': np.sqrt(fold_mse)
            for fold_number, fold_mse in fold_blend.items()
        }
    })
blend_summary = pd.DataFrame(blend_rows)
print("\n🧪 Mélange en espace log : poids testés plus fins autour de la référence 75 % XGBoost :")
display(blend_summary)
print("Compare le RMSLE global et les gains par fenêtre face à 75 %. Ne change pas le poids final sur le seul meilleur score OOF : ces mêmes fenêtres servent à comparer les poids.")

from sklearn.metrics import brier_score_loss
calibration_rows = []
for fold_number, fold_data in extended_oof.groupby('fold'):
    calibration_rows.append({
        'fold': fold_number,
        'n': len(fold_data),
        'taux_positif_reel': fold_data['actual_nonzero'].mean(),
        'probabilite_moyenne': fold_data['probability_nonzero'].mean(),
        'brier': brier_score_loss(
            fold_data['actual_nonzero'], fold_data['probability_nonzero']
        )
    })
calibration_by_fold = pd.DataFrame(calibration_rows)
print("\n🎯 Brier score du classifieur XGBoost par fenêtre (plus bas = meilleur) :")
display(calibration_by_fold)
print(
    "Brier global : "
    f"{brier_score_loss(extended_oof['actual_nonzero'], extended_oof['probability_nonzero']):.6f}"
)
print(
    "Itérations fixes (régression directe, classifieur, régresseur conditionnel) : "
    f"{extended_baseline_iterations}, {extended_classifier_iterations}, "
    f"{extended_regressor_iterations}"
)
print("Cette validation historique élargit le diagnostic ; elle ne remplace pas une période finale tenue à part.")

## 8. Entraînement final et export de la soumission

Réentraîne les modèles sur toutes les observations étiquetées et écrit une soumission séparée. Le poids retenu est 75 % XGBoost deux étapes et 25 % LightGBM direct.

In [ ]:
print("🔮 Entraînement final du mélange LightGBM CPU + XGBoost GPU...")

final_train_data = df.loc[df['sales'].notna()].copy()
final_test_data = df.loc[df['sales'].isna()].sort_values('id').copy()
if final_train_data.empty or final_test_data.empty:
    raise ValueError("Les données d'entraînement ou de test sont vides.")
if final_train_data['sales'].lt(0).any():
    raise ValueError("La cible sales contient des valeurs négatives inattendues.")

final_X_train = final_train_data[features]
final_X_test = final_test_data[features].copy()
for column in cat_cols:
    final_X_test[column] = pd.Categorical(
        final_X_test[column], categories=df[column].cat.categories
    )

final_y_nonzero = final_train_data['sales'].gt(0).astype('int8')
final_positive_mask = final_train_data['sales'].gt(0)
if final_y_nonzero.nunique() != 2 or not final_positive_mask.any():
    raise ValueError("Le jeu d'entraînement final ne contient pas les classes nécessaires.")

final_classifier_iterations = 1075
final_regressor_iterations = 3707
final_direct_model = lgb.LGBMRegressor(
    objective='regression_l2',
    metric='rmse',
    boosting_type='gbdt',
    n_estimators=2918,
    learning_rate=0.025,
    num_leaves=63,
    max_depth=8,
    min_child_samples=150,
    subsample=0.75,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
final_direct_model.fit(final_X_train, final_train_data['sales_log'])
final_direct_log_prediction = np.maximum(
    final_direct_model.predict(final_X_test), 0
)

final_classifier = xgb.XGBClassifier(
    **XGB_COMMON_PARAMS,
    n_estimators=final_classifier_iterations,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)
final_classifier.fit(final_X_train, final_y_nonzero)

final_regressor = xgb.XGBRegressor(
    **XGB_COMMON_PARAMS,
    n_estimators=final_regressor_iterations,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42
)
final_regressor.fit(
    final_X_train.loc[final_positive_mask],
    final_train_data.loc[final_positive_mask, 'sales_log']
)
final_classifier.set_params(device='cpu')
final_regressor.set_params(device='cpu')
final_probability_nonzero = final_classifier.predict_proba(final_X_test)[:, 1]
final_conditional_log_sales = np.maximum(final_regressor.predict(final_X_test), 0)
final_two_stage_log_prediction = final_probability_nonzero * final_conditional_log_sales

final_blend_two_stage_weight = 0.75
final_blend_log_prediction = (
    (1 - final_blend_two_stage_weight) * final_direct_log_prediction
    + final_blend_two_stage_weight * final_two_stage_log_prediction
)
final_sales_prediction = np.maximum(np.expm1(final_blend_log_prediction), 0)
if not np.isfinite(final_sales_prediction).all():
    raise ValueError("Le modèle a produit des prédictions non finies.")

final_submission = pd.DataFrame({
    'id': final_test_data['id'].to_numpy(),
    'sales': final_sales_prediction
})
expected_test_ids = test.sort_values('id')['id'].to_numpy()
if (
    len(final_submission) != len(test)
    or not np.array_equal(final_submission['id'].to_numpy(), expected_test_ids)
):
    raise ValueError("Les lignes prédites ne correspondent pas aux identifiants du test Kaggle.")

output_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
final_submission_path = os.path.join(output_dir, 'submission_portfolio_xgb75_lgbm25.csv')
final_submission.to_csv(final_submission_path, index=False)
print(f"✅ Soumission créée : {final_submission_path}")
print(f"Lignes : {len(final_submission):,} | Poids XGBoost deux étapes : "
      f"{final_blend_two_stage_weight:.0%} | Itérations classifieur/régresseur : "
      f"{final_classifier_iterations}/{final_regressor_iterations}")
print(final_submission.head(10))